# Burgers front diagnostics — GPU-only score pass

Read the ten saved validation fields from the completed Burgers kernel and measure final-time shock position and reference-scaled 10–90% width on two Tesla T4s. No model is loaded and no training occurs.


In [ ]:
import csv, json, time, statistics
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch

START=time.perf_counter()
OUT=Path('/kaggle/working/burgers-front-diagnostics'); OUT.mkdir(parents=True,exist_ok=True)
if not torch.cuda.is_available() or torch.cuda.device_count()!=2:
    raise RuntimeError('Exactly two CUDA devices are required; CPU fallback is disabled.')
gpus=[torch.cuda.get_device_name(i) for i in range(2)]
if any('T4' not in name.upper() for name in gpus): raise RuntimeError(f'Expected two Tesla T4s, found {gpus}')

import re
items=[]
for p in Path('/kaggle/input').rglob('seed-*.npz'):
    match=re.fullmatch(r'seed-(811|812|813|814|815)-(rbf|vanilla)\.npz',p.name)
    if match:
        items.append({'path':p,'seed':int(match.group(1)),'arm':match.group(2)})
if len(items)!=10 or sorted({x['seed'] for x in items})!=[811,812,813,814,815]:
    raise RuntimeError(f'Expected ten saved endpoint files; found {len(items)} at {[str(p) for p in Path("/kaggle/input").rglob("*.npz")]}')

def crossing(x,y,level):
    idx=torch.where((y[:-1]>=level)&(y[1:]<level))[0]
    if idx.numel()==0: return None
    x0=x[idx]; x1=x[idx+1]; y0=y[idx]; y1=y[idx+1]
    loc=x0+(level-y0)*(x1-x0)/(y1-y0)
    return float(loc[torch.argmin(torch.abs(loc))].item())

def score(item,device_index):
    device=torch.device(f'cuda:{device_index}')
    with np.load(item['path'],allow_pickle=False) as z:
        x=torch.as_tensor(z['x'].copy(),device=device,dtype=torch.float64)
        pred=torch.as_tensor(z['prediction'].copy(),device=device,dtype=torch.float64)[:,-1]
        ref=torch.as_tensor(z['reference'].copy(),device=device,dtype=torch.float64)[:,-1]
        residual=torch.as_tensor(z['residual'].copy(),device=device,dtype=torch.float64)[:,-1]
    region=(x>=-0.4)&(x<=-0.2); right=(x>=0.2)&(x<=0.4)
    if not bool(region.any()) or not bool(right.any()): raise RuntimeError('Reference plateau windows are empty')
    left_state=torch.median(ref[region]); right_state=torch.median(ref[right]); jump=left_state-right_state
    high=right_state+0.9*jump; low=right_state+0.1*jump
    ref_x0=crossing(x,ref,torch.zeros((),device=device,dtype=x.dtype))
    pred_x0=crossing(x,pred,torch.zeros((),device=device,dtype=x.dtype))
    ref_hi=crossing(x,ref,high); ref_lo=crossing(x,ref,low)
    pred_hi=crossing(x,pred,high); pred_lo=crossing(x,pred,low)
    if any(v is None for v in [ref_x0,pred_x0,ref_hi,ref_lo,pred_hi,pred_lo]):
        return {'seed':item['seed'],'arm':item['arm'],'status':'front_crossing_missing','source_npz':str(item['path'])}
    width_ref=ref_lo-ref_hi; width_pred=pred_lo-pred_hi
    mask=torch.abs(x-ref_x0)<=0.2
    err=pred[mask]-ref[mask]
    torch.cuda.synchronize(device)
    return {'seed':item['seed'],'arm':item['arm'],'status':'complete','device':gpus[device_index],
        'cuda_device_index':device_index,'reference_front_x_zero':ref_x0,'predicted_front_x_zero':pred_x0,
        'front_position_abs_error':abs(pred_x0-ref_x0),'reference_front_width_10_90':width_ref,
        'predicted_front_width_10_90':width_pred,'front_width_abs_error':abs(width_pred-width_ref),
        'front_region_x_halfwidth':0.2,'front_region_field_rmse':float(torch.sqrt(torch.mean(err.square())).item()),
        'front_region_positive_deviation':max(0.0,float(torch.max(err).item())),
        'front_region_negative_deviation':max(0.0,float(-torch.min(err).item())),
        'front_region_residual_rms':float(torch.sqrt(torch.mean(residual[mask].square())).item()),
        'threshold_high':float(high.item()),'threshold_low':float(low.item()),'source_npz':str(item['path'])}

by_seed={s:[] for s in [811,812,813,814,815]}
for seed in sorted(by_seed):
    pair=sorted([x for x in items if x['seed']==seed],key=lambda x:x['arm'])
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures=[pool.submit(score,x,i) for i,x in enumerate(pair)]
        by_seed[seed]=[f.result() for f in futures]
rows=[r for seed in sorted(by_seed) for r in by_seed[seed]]
if any(r['status']!='complete' for r in rows): raise RuntimeError('At least one front diagnostic is incomplete')
summary={'status':'front_diagnostics_complete','source_kernel':'rodrimandub/af-pikans-burgers-rbf-vanilla-paired-20260927-r2',
    'problem':'Burgers PINNacle 1D; final t snapshot; zero crossing and reference-defined 10-90% shock width',
    'n_per_arm':5,'reference':'saved CUDA Cole-Hopf/Gauss-Hermite order 128 field on the common grid',
    'threshold_method':'plateau medians on x in [-0.4,-0.2] and [0.2,0.4]; prediction uses reference levels',
    'front_region':'|x-reference front position| <= 0.2','results':rows,'gpus':gpus,
    'wall_seconds':time.perf_counter()-START,'training_performed':False,'scientific_accepted':False}
(OUT/'front_diagnostics.json').write_text(json.dumps(summary,indent=2,sort_keys=True,allow_nan=False)+'\n',encoding='utf-8')
cols=list(rows[0].keys())
with (OUT/'front_diagnostics.csv').open('w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f,fieldnames=cols,extrasaction='ignore'); w.writeheader(); w.writerows(rows)
for arm in ['rbf','vanilla']:
    a=[r for r in rows if r['arm']==arm]
    print(arm,'mean_abs_front_position_error',statistics.mean(r['front_position_abs_error'] for r in a),
        'mean_abs_front_width_error',statistics.mean(r['front_width_abs_error'] for r in a),
        'mean_front_field_rmse',statistics.mean(r['front_region_field_rmse'] for r in a))
print(json.dumps({'status':summary['status'],'wall_seconds':summary['wall_seconds'],'gpus':gpus,'training_performed':False},indent=2))
